<div style="border: 5px solid black; padding: 10px;">

# **Name:** Tim Hollis  
# **Course:** DSC540 - Data Preparation  
# **Date:** 02/28/2026  
# **Assignment:** Milestone 4 – Connecting to an API / Pulling, Cleaning & Formatting Data

</div>

### Initial Setup:

In [1]:
# Load Libraries
import requests
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Load API Key
with open('census_config.json') as f:
    config = json.load(f)

API_KEY = config['api_key']

# API Configuration
BASE_URL = 'https://api.census.gov/data/2021/acs/acs1/profile'

VARIABLES = {
    'DP05_0001E': 'total_population',
    'DP03_0062E': 'median_household_income',
    'DP03_0025E': 'mean_commute_time_min',
    'DP03_0128PE': 'poverty_rate_pct',
    'DP05_0018E': 'median_age',
    'DP04_0134E': 'median_gross_rent'
}

PARAMS = {
    'get': f'NAME,{",".join(VARIABLES.keys())}',
    'for': 'place:*',
    'in': 'state:*',
    'key': API_KEY
}

# Pull Data from API
response = requests.get(BASE_URL, params=PARAMS)
raw_data = response.json()


# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.width', 1000)

# Preview data
print(f'✅ API connection successful!')
print(f'📦 Raw records returned: {len(raw_data) - 1:,}')
print(f'📋 Columns returned: {len(raw_data[0])}')
print(f'\n🔍 First row (headers): {raw_data[0]}')
print(f'🔍 Second row (sample): {raw_data[1]}')

✅ API connection successful!
📦 Raw records returned: 634
📋 Columns returned: 9

🔍 First row (headers): ['NAME', 'DP05_0001E', 'DP03_0062E', 'DP03_0025E', 'DP03_0128PE', 'DP05_0018E', 'DP04_0134E', 'state', 'place']
🔍 Second row (sample): ["O'Fallon city, Missouri", '93651', '97868', '23.2', '5.9', '35.8', '1200', '29', '54074']


## Assignment Overview

Checkpoint 4 requires connecting to an API data source, pulling the raw JSON response, 
and performing at least 5 transformation and cleansing steps directly against the data. 
All work is done in-memory against the live API response.

**Data Source:** U.S. Census Bureau — American Community Survey (ACS) 1-Year Estimates (2021)  
**API Endpoint:** [Census ACS 1-Year Profile API](https://api.census.gov/data/2021/acs/acs1/profile)  
**Geographic Level:** All U.S. places (cities/towns) with 65,000+ population (~634 records)  

**Variables Pulled:**
| Census Code | Description |
|---|---|
| `DP05_0001E` | Total Population |
| `DP03_0062E` | Median Household Income |
| `DP03_0025E` | Mean Commute Time (minutes) |
| `DP03_0128PE` | Poverty Rate (%) |
| `DP05_0018E` | Median Age |
| `DP04_0134E` | Median Gross Rent |

**Transformation Steps:**
| Step | Description |
|---|---|
| 1 | Replace cryptic API headers with human-readable column names |
| 2 | Convert numeric fields from strings to proper numeric data types |
| 3 | Replace Census null sentinel values (-999999999) with NaN |
| 4 | Split the NAME field into separate City and State columns and fix casing |
| 5 | Remove the word "city", "town", "village" etc. from city names for future merging |
| 6 | Flag and review outliers in income and rent columns |

---

## Data Source

The data for this checkpoint is pulled live from the **U.S. Census Bureau's American 
Community Survey (ACS) 1-Year Estimates API** for the year 2021. The ACS is conducted 
annually by the Census Bureau and provides detailed demographic, social, economic, and 
housing characteristics for U.S. communities.

**Why this source?**  
The ACS API returns data in JSON format directly from the Census Bureau's servers, 
making it ideal for practicing in-memory data transformation without relying on 
pre-exported files. It also connects naturally to the other datasets in this project — 
Walk Score city rankings and HUD Fair Market Rent data — through shared city-level 
geographic identifiers.

**API Access:**  
- No cost, publicly available with a free API key  
- Key stored externally in `census_config.json` — never hardcoded  
- Documentation: [Census ACS API Docs](https://www.census.gov/data/developers/data-sets/acs-1year.html)  

**Credibility:**  
The Census Bureau is a federal statistical agency and one of the most authoritative 
sources of demographic and socioeconomic data in the United States. ACS estimates are 
based on continuous surveying throughout the year and are widely used in academic, 
government, and industry research.

**Limitations:**  
ACS 1-year estimates are only available for geographies with populations of 65,000 or 
more, which means smaller cities and rural areas are not represented in this dataset.

---

## Step 1 — Replace Headers

The Census API returns cryptic variable codes as column names (e.g. `DP05_0001E`). 
This step converts the raw JSON response into a DataFrame and immediately renames 
all columns to human-readable labels for clarity and usability.

---

In [2]:
# Convert JSON response to DataFrame using first row as headers
df = pd.DataFrame(raw_data[1:], columns=raw_data[0])

print('📋 Original column names from API:')
print(df.columns.tolist())

# Build map of variables and appended API fields
rename_map = {
    'NAME': 'full_name',
    'DP05_0001E': 'total_population',
    'DP03_0062E': 'median_household_income',
    'DP03_0025E': 'mean_commute_time_min',
    'DP03_0128PE': 'poverty_rate_pct',
    'DP05_0018E': 'median_age',
    'DP04_0134E': 'median_gross_rent',
    'state': 'state_fips',
    'place': 'place_fips'
}

df.rename(columns=rename_map, inplace=True)

print(f'\n✅ Columns after renaming:')
print(df.columns.tolist())
print(f'\n🔍 Sample row:')
display(df.head(3))

📋 Original column names from API:
['NAME', 'DP05_0001E', 'DP03_0062E', 'DP03_0025E', 'DP03_0128PE', 'DP05_0018E', 'DP04_0134E', 'state', 'place']

✅ Columns after renaming:
['full_name', 'total_population', 'median_household_income', 'mean_commute_time_min', 'poverty_rate_pct', 'median_age', 'median_gross_rent', 'state_fips', 'place_fips']

🔍 Sample row:


,full_name,total_population,median_household_income,mean_commute_time_min,poverty_rate_pct,median_age,median_gross_rent,state_fips,place_fips
0,"O'Fallon city, Missouri",93651,97868,23.2,5.9,35.8,1200,29,54074
1,"St. Louis city, Missouri",293310,49965,20.8,20.4,36.8,843,29,65000
2,"Passaic city, New Jersey",69637,51806,24.4,23.0,30.8,1256,34,56550


---

## Step 2 — Convert Numeric Fields from Strings to Proper Data Types

The Census API returns all values as strings regardless of their actual data type. 
This step converts numeric columns to their appropriate types so they can be used 
in calculations, comparisons, and visualizations. Non-convertible values are 
coerced to NaN rather than throwing errors, which sets us up cleanly for Step 3.

---

In [3]:
# Determine which columns should be numeric and their target types
numeric_cols = {
    'total_population': 'int',
    'median_household_income': 'int',
    'median_gross_rent': 'int',
    'mean_commute_time_min': 'float',
    'poverty_rate_pct': 'float',
    'median_age': 'float'
}

# Check dtypes before conversion
print('📋 Data types BEFORE conversion:')
print(df[list(numeric_cols.keys())].dtypes)

# Convert each column compatible to numeric and force others to NaN
# compatible value
for col, dtype in numeric_cols.items():
    df[col] = pd.to_numeric(df[col], errors='coerce')
    if dtype == 'int':
        df[col] = df[col]


# Confirm dtypes after conversion
print(f'\n✅ Data types AFTER conversion:')
print(df[list(numeric_cols.keys())].dtypes)

# Preview data after conversion
print(f'\n🔍 Sample after conversion:')
display(df[['full_name'] + list(numeric_cols.keys())].head(3))

📋 Data types BEFORE conversion:
total_population           str
median_household_income    str
median_gross_rent          str
mean_commute_time_min      str
poverty_rate_pct           str
median_age                 str
dtype: object

✅ Data types AFTER conversion:
total_population             int64
median_household_income      int64
median_gross_rent            int64
mean_commute_time_min      float64
poverty_rate_pct           float64
median_age                 float64
dtype: object

🔍 Sample after conversion:


,full_name,total_population,median_household_income,median_gross_rent,mean_commute_time_min,poverty_rate_pct,median_age
0,"O'Fallon city, Missouri",93651,97868,1200,23.2,5.9,35.8
1,"St. Louis city, Missouri",293310,49965,843,20.8,20.4,36.8
2,"Passaic city, New Jersey",69637,51806,1256,24.4,23.0,30.8


---

## Step 3 — Replace Census Null Sentinel Values with NaN

The Census Bureau uses -999999999 as a placeholder for suppressed or unavailable 
data rather than leaving fields empty. Now that our columns are numeric, we can 
identify and replace these sentinel values with proper NaN so they are treated as 
missing data in all future calculations and visualizations.

---

In [4]:
# Define sentinel value used by Census Bureau for suppressed data
CENSUS_NULL = -999999999

# Check for sentinel values before replacement
print('🔍 Sentinel values found BEFORE replacement:')
numeric_cols = [
    'total_population',
    'median_household_income',
    'median_gross_rent',
    'mean_commute_time_min',
    'poverty_rate_pct',
    'median_age'
]

for col in numeric_cols:
    count = (df[col] == CENSUS_NULL).sum()
    if count > 0:
        print(f'   ⚠️  {col}: {count:,} sentinel value(s) found')
    else:
        print(f'   ✅ {col}: clean')

# Replace all sentinel values with NaN
df[numeric_cols] = df[numeric_cols].replace(CENSUS_NULL, np.nan)

# Confirm replacement and summarize missing data
print(f'\n✅ Sentinel values replaced. Missing value summary:')
missing = df[numeric_cols].isna().sum()
missing_pct = (df[numeric_cols].isna().mean() * 100).round(1)

summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

print(summary)
print(f'\n📦 Total records in dataset: {len(df):,}')

🔍 Sentinel values found BEFORE replacement:
   ✅ total_population: clean
   ✅ median_household_income: clean
   ✅ median_gross_rent: clean
   ⚠️  mean_commute_time_min: 2 sentinel value(s) found
   ✅ poverty_rate_pct: clean
   ✅ median_age: clean

✅ Sentinel values replaced. Missing value summary:
                         missing_count  missing_pct
total_population                     0          0.0
median_household_income              0          0.0
median_gross_rent                    0          0.0
mean_commute_time_min                2          0.3
poverty_rate_pct                     0          0.0
median_age                           0          0.0

📦 Total records in dataset: 634


---

## Step 4 — Split Full Name Field into Separate City and State Columns

The Census API returns geography as a single combined field (e.g. `"O'Fallon city, 
Missouri"`). This step splits that field into separate `city_raw` and `state_name` 
columns, which is required to build the standardized merge key that will connect 
this dataset to the Walk Score and HUD FMR tables in the final milestone.

---

In [5]:
# Preview the variety of formats before splitting
print('🔍 Sample of full_name formats before splitting:')
print(df['full_name'].head(10).tolist())

# Split on the last comma only
df[['city_raw', 'state_name']] = df['full_name'].str.rsplit(
    ',', n=1, expand=True)

# Strip all whitespace from both new columns
df['city_raw'] = df['city_raw'].str.strip()
df['state_name'] = df['state_name'].str.strip()

# Confirm the split worked correctly
print(f'\n✅ Split complete. Sample of city_raw and state_name:')
display(df[['full_name', 'city_raw', 'state_name']].head(10))

# Check for any rows where split may have failed
nulls = df[['city_raw', 'state_name']].isna().sum()
print(f'\n🔍 Null check after split:')
print(nulls)

🔍 Sample of full_name formats before splitting:
["O'Fallon city, Missouri", 'St. Louis city, Missouri', 'Passaic city, New Jersey', 'Nashua city, New Hampshire', 'Rochester city, Minnesota', 'Paterson city, New Jersey', 'Warren city, Michigan', "Lee's Summit city, Missouri", 'Gastonia city, North Carolina', 'Minneapolis city, Minnesota']

✅ Split complete. Sample of city_raw and state_name:


,full_name,city_raw,state_name
0,"O'Fallon city, Missouri",O'Fallon city,Missouri
1,"St. Louis city, Missouri",St. Louis city,Missouri
2,"Passaic city, New Jersey",Passaic city,New Jersey
3,"Nashua city, New Hampshire",Nashua city,New Hampshire
4,"Rochester city, Minnesota",Rochester city,Minnesota
5,"Paterson city, New Jersey",Paterson city,New Jersey
6,"Warren city, Michigan",Warren city,Michigan
7,"Lee's Summit city, Missouri",Lee's Summit city,Missouri
8,"Gastonia city, North Carolina",Gastonia city,North Carolina
9,"Minneapolis city, Minnesota",Minneapolis city,Minnesota



🔍 Null check after split:
city_raw      0
state_name    0
dtype: int64


---

## Step 5 — Strip Place Type Suffixes and Build Standardized Merge Key

The `city_raw` column still contains place type descriptors appended by the Census 
Bureau (e.g. "city", "town", "village", "borough"). This step removes those suffixes 
and converts full state names to standard 2-letter abbreviations, then combines them 
into a single `merge_key` column formatted as `"City, ST"`. This key will be used to 
join this dataset with the Walk Score and HUD FMR tables in the final milestone.

---

In [6]:
# Define all Census "place" type suffixes to remove
place_suffixes = [
    ' city', ' town', ' village', ' borough', ' municipality',
    ' city and borough', ' unified government', ' consolidated government',
    ' metropolitan government', ' urban county', ' charter township',
    ' township', ' plantation', ' gore', ' grant'
]

# Remove suffixes - sorted to avpid accidental removal based on fuzzy search
place_suffixes_sorted = sorted(place_suffixes, key=len, reverse=True)

df['city'] = df['city_raw'].str.strip()

for suffix in place_suffixes_sorted:
    mask = df['city'].str.endswith(suffix, na=False)
    df.loc[mask, 'city'] = df.loc[mask, 'city'].str[:-len(suffix)].str.strip()

print('🔍 Sample of city_raw vs city after suffix removal:')
display(df[['city_raw', 'city']].head(15))

# Build state name to abbreviation mapping
state_abbr_map = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'Delaware': 'DE',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC',
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY',
    'District of Columbia': 'DC',
    'Puerto Rico': 'PR'}

# Map full state names to abbreviations
df['state'] = df['state_name'].map(state_abbr_map)

# Check for any states that didn't map
unmapped = df[df['state'].isna()]['state_name'].unique()
if len(unmapped) > 0:
    print(f'\n⚠️  Unmapped states found: {unmapped}')
else:
    print(f'\n✅ All states mapped successfully')

# Build the merge key
df['merge_key'] = df['city'] + ', ' + df['state']

# Final confirmation
print(f'\n✅ Merge key sample:')
display(df[['city_raw', 'city', 'state_name', 'state', 'merge_key']].head(10))

🔍 Sample of city_raw vs city after suffix removal:


,city_raw,city
0,O'Fallon city,O'Fallon
1,St. Louis city,St. Louis
2,Passaic city,Passaic
3,Nashua city,Nashua
4,Rochester city,Rochester
5,Paterson city,Paterson
6,Warren city,Warren
7,Lee's Summit city,Lee's Summit
8,Gastonia city,Gastonia
9,Minneapolis city,Minneapolis



✅ All states mapped successfully

✅ Merge key sample:


,city_raw,city,state_name,state,merge_key
0,O'Fallon city,O'Fallon,Missouri,MO,"O'Fallon, MO"
1,St. Louis city,St. Louis,Missouri,MO,"St. Louis, MO"
2,Passaic city,Passaic,New Jersey,NJ,"Passaic, NJ"
3,Nashua city,Nashua,New Hampshire,NH,"Nashua, NH"
4,Rochester city,Rochester,Minnesota,MN,"Rochester, MN"
5,Paterson city,Paterson,New Jersey,NJ,"Paterson, NJ"
6,Warren city,Warren,Michigan,MI,"Warren, MI"
7,Lee's Summit city,Lee's Summit,Missouri,MO,"Lee's Summit, MO"
8,Gastonia city,Gastonia,North Carolina,NC,"Gastonia, NC"
9,Minneapolis city,Minneapolis,Minnesota,MN,"Minneapolis, MN"


---

## Step 6 — Identify Outliers in Income and Rent Columns

Outliers in socioeconomic data like median household income and median gross rent 
can represent genuine extremes (e.g. very wealthy cities or high cost-of-living 
markets) or potential data quality issues. This step uses the IQR method to flag 
statistical outliers, reviews them for legitimacy, and documents findings without 
removing any records — preserving the integrity of the original data.

---

In [7]:
# Define columns to check for outliers
outlier_cols = ['median_household_income', 'median_gross_rent']

# Calculate IQR bounds and flag outliers for each column
print('📊 Outlier Analysis Using IQR Method:')
print('=' * 55)

outlier_flags = pd.DataFrame(index=df.index)

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_flags[f'{col}_outlier'] = (
        (df[col] < lower_bound) | (df[col] > upper_bound)
    )

    print(f'\n📌 {col}:')
    print(f'   Q1: ${Q1:,.0f}  |  Q3: ${Q3:,.0f}  |  IQR: ${IQR:,.0f}')
    print(f'   Lower bound: ${lower_bound:,.0f}')
    print(f'   Upper bound: ${upper_bound:,.0f}')
    print(f'   Outliers found: {len(outliers):,} records')

    print(f'\n   🔍 Top 5 high outliers:')
    display(
        outliers.nlargest(5, col)[['merge_key', col]]
        .assign(**{col: lambda x: x[col].map('${:,.0f}'.format)})
    )

# Flag any city that is an outlier in either column
df['is_outlier'] = outlier_flags.any(axis=1)

print(
    f'\n✅ Total records flagged as outliers in any column: {df["is_outlier"].sum():,}')
print(f'📦 These records are FLAGGED only — no data has been removed')

📊 Outlier Analysis Using IQR Method:

📌 median_household_income:
   Q1: $55,282  |  Q3: $88,223  |  IQR: $32,940
   Lower bound: $5,872
   Upper bound: $137,634
   Outliers found: 17 records

   🔍 Top 5 high outliers:


,merge_key,median_household_income
274,"Dublin, CA","$205,219"
77,"Sammamish, WA","$201,370"
553,"Palo Alto, CA","$195,781"
365,"Newton, MA","$183,208"
387,"Milpitas, CA","$169,460"



📌 median_gross_rent:
   Q1: $1,032  |  Q3: $1,660  |  IQR: $628
   Lower bound: $90
   Upper bound: $2,603
   Outliers found: 9 records

   🔍 Top 5 high outliers:


,merge_key,median_gross_rent
553,"Palo Alto, CA","$3,063"
410,"Redwood City, CA","$2,950"
274,"Dublin, CA","$2,852"
313,"San Ramon, CA","$2,848"
425,"Newport Beach, CA","$2,815"



✅ Total records flagged as outliers in any column: 19
📦 These records are FLAGGED only — no data has been removed


---

## Final Human-Readable Dataset

All transformation steps are complete. The dataset below reflects the fully cleaned 
and standardized Census ACS data ready for future merging with the Walk Score and 
HUD Fair Market Rent tables. Outlier records are flagged but retained.

---

In [8]:
# Select and order final columns for presentation
final_cols = [
    'merge_key',
    'city',
    'state',
    'total_population',
    'median_household_income',
    'median_gross_rent',
    'mean_commute_time_min',
    'poverty_rate_pct',
    'median_age',
    'state_fips',
    'place_fips',
    'is_outlier'
]

df_final = df[final_cols].copy()

# Sort by state then city for readability
df_final = df_final.sort_values(['state', 'city']).reset_index(drop=True)

# Print summary statistics
print('📊 Final Dataset Summary:')
print(f'   Total records      : {len(df_final):,}')
print(f'   Total columns      : {len(df_final.columns):,}')
print(f'   States represented : {df_final["state"].nunique():,}')
print(f'   Outliers flagged   : {df_final["is_outlier"].sum():,}')
print(
    f'\n   💰 Median Household Income — Avg : ${df_final["median_household_income"].mean():,.0f}')
print(
    f'   💰 Median Household Income — Min : ${df_final["median_household_income"].min():,.0f}')
print(
    f'   💰 Median Household Income — Max : ${df_final["median_household_income"].max():,.0f}')
print(
    f'\n   🏠 Median Gross Rent — Avg       : ${df_final["median_gross_rent"].mean():,.0f}')
print(
    f'   🏠 Median Gross Rent — Min       : ${df_final["median_gross_rent"].min():,.0f}')
print(
    f'   🏠 Median Gross Rent — Max       : ${df_final["median_gross_rent"].max():,.0f}')
print(
    f'\n   🚗 Mean Commute Time — Avg       : {df_final["mean_commute_time_min"].mean():,.1f} min')
print(
    f'   📉 Poverty Rate — Avg            : {df_final["poverty_rate_pct"].mean():,.1f}%')

# Step 4 — Display final dataset
print(f'\n✅ Final cleaned dataset (sorted by state, then city):')
display(df_final)

📊 Final Dataset Summary:
   Total records      : 634
   Total columns      : 12
   States represented : 50
   Outliers flagged   : 19

   💰 Median Household Income — Avg : $74,457
   💰 Median Household Income — Min : $17,207
   💰 Median Household Income — Max : $205,219

   🏠 Median Gross Rent — Avg       : $1,388
   🏠 Median Gross Rent — Min       : $530
   🏠 Median Gross Rent — Max       : $3,063

   🚗 Mean Commute Time — Avg       : 24.5 min
   📉 Poverty Rate — Avg            : 14.0%

✅ Final cleaned dataset (sorted by state, then city):


,merge_key,city,state,total_population,median_household_income,median_gross_rent,mean_commute_time_min,poverty_rate_pct,median_age,state_fips,place_fips,is_outlier
0,"Anchorage, AK",Anchorage,AK,288121,86654,1335,19.1,9.1,35.2,02,03000,False
1,"Auburn, AL",Auburn,AL,78552,48531,1009,21.0,24.8,27.8,01,03076,False
2,"Birmingham, AL",Birmingham,AL,196410,36614,895,21.3,28.3,37.2,01,07000,False
3,"Dothan, AL",Dothan,AL,71283,45088,832,21.1,21.3,40.5,01,21184,False
4,"Hoover, AL",Hoover,AL,92588,99276,1212,24.2,6.0,38.9,01,35896,False
...,...,...,...,...,...,...,...,...,...,...,...,...
629,"Milwaukee, WI",Milwaukee,WI,569326,46637,935,21.4,23.8,32.3,55,53000,False
630,"Oshkosh, WI",Oshkosh,WI,66594,55446,755,20.1,17.9,35.3,55,60500,False
631,"Racine, WI",Racine,WI,77131,47861,862,21.2,22.6,34.6,55,66000,False
632,"Waukesha, WI",Waukesha,WI,71254,69533,1032,20.1,8.1,36.2,55,84250,False


In [9]:
# Check how many sentinel-like values exist
bad_commute = df_final[df_final['mean_commute_time_min'] < 0]
print(f'⚠️  Negative commute time records found: {len(bad_commute):,}')
print(bad_commute[['merge_key', 'mean_commute_time_min']])

# Replace any negative commute values with NaN
df_final['mean_commute_time_min'] = df_final['mean_commute_time_min'].apply(
    lambda x: np.nan if x < 0 else x
)

# Reprint corrected average
print(
    f'\n✅ Corrected Mean Commute Time — Avg: {df_final["mean_commute_time_min"].mean():,.1f} min')

⚠️  Negative commute time records found: 0
Empty DataFrame
Columns: [merge_key, mean_commute_time_min]
Index: []

✅ Corrected Mean Commute Time — Avg: 24.5 min


<div style="border: 5px solid black; padding: 10px;">

## Transformation Summary

The table below summarizes the number of changes made during each transformation step,
what was changed, and the reason for the change.

| Step | Description | What Changed | Count | Why |
|---|---|---|---|---|
| 1 | Replace headers | Column names renamed from Census variable codes to readable labels | 9 columns | Cryptic API codes are not human-readable or audit-friendly |
| 2 | Convert data types | Numeric columns converted from string to int64/float64 | 6 columns | API returns all values as strings regardless of data type |
| 3 | Replace sentinel values | `-999999999` replaced with `NaN` in commute time column | 2 records | Census uses sentinel integers instead of null for suppressed data |
| 4 | Split name field | `full_name` split into `city_raw` and `state_name` | 634 records | City and state were combined in a single field unusable for merging |
| 5 | Build merge key | Place suffixes stripped, state names converted to abbreviations, `merge_key` created | 634 records | Standardized `City, ST` format required to join across all three datasets |
| 6 | Flag outliers | Records flagged with `is_outlier` column in income and rent fields | 19 records flagged, 0 removed | Extreme values identified for transparency but retained as legitimate data |

## Ethical Implications of Data Wrangling

The transformations applied to the U.S. Census Bureau ACS API dataset were designed 
to improve clarity, consistency, and analytical usability while preserving the 
integrity of the original data. The changes made included renaming column headers, 
converting data types, replacing sentinel null values, splitting and standardizing 
geographic fields, and flagging statistical outliers. No records were deleted and no 
values were altered beyond formatting and type conversion.

From a legal and regulatory standpoint, the Census Bureau's ACS data is publicly 
available under Title 13 of the U.S. Code, which governs Census data collection and 
confidentiality. While the data is aggregated and contains no personally identifiable 
information, it is still subject to statistical disclosure limitations — which is 
exactly why sentinel values like -999999999 appear in the dataset for suppressed 
fields. This was observed in the `mean_commute_time_min` column for two Census 
Designated Places (CDPs) where commute data was statistically unreliable.

Several assumptions were made during transformation. Stripping place type suffixes 
such as "city", "town", and "village" from city names assumes that these distinctions 
are not analytically meaningful for the purposes of this project. Additionally, 
retaining outlier records rather than removing them assumes that extreme values like 
Dublin, CA's median household income of `$205,219` or Palo Alto, CA's median rent of 
`$3,063` represent genuine socioeconomic conditions rather than data errors. Both 
assumptions are documented and defensible given the source credibility of the Census 
Bureau.

A meaningful ethical risk in this dataset is its geographic scope. Because ACS 
1-year estimates are only published for places with populations of 65,000 or more, 
smaller cities, rural communities, and many minority-majority communities are entirely 
excluded from this analysis. This limitation could lead to conclusions that reflect 
the experiences of larger, more urban populations while underrepresenting communities 
that may face the greatest transportation and affordability challenges. To mitigate 
this risk, all findings from this analysis will be framed within the explicit 
limitation that results apply only to larger U.S. cities and should not be 
generalized to the broader U.S. population.

The data was sourced directly from the Census Bureau's public API, acquired without 
scraping restrictions or terms of service violations, and accessed using a registered 
API key. The Census Bureau is one of the most credible and methodologically rigorous 
statistical agencies in the United States, providing strong assurance of data quality 
and collection integrity.

<div style="border: 5px solid black; padding: 10px;">

# **Name:** Tim Hollis  
# **Course:** DSC540 - Data Preparation  
# **Date:**  02/15/2026
# **Assignment:** Milestone 3
## Walk Score City Rankings
## Data Source: 'https://www.walkscore.com/cities-and-neighborhoods/'  


### **Task**: Perform at least 5 transformation/cleansing steps directly against the HTML source to produce a clean, human-readable Dataframe ready for future merging.  

### Initial Setup

In [11]:
# Loading Libraries
import pandas as pd
import requests
from io import StringIO
from IPython.display import display

# Setting display for consistent output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 15)
pd.set_option('display.float_format', '{:.1f}'.format)

# Load data directly from HTML
url = 'https://www.walkscore.com/cities-and-neighborhoods/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

response = requests.get(url, headers=headers)

# Wrapping for directing to memory and not file path
tables = pd.read_html(StringIO(response.text))

print(f"Found {len(tables)} table(s) on the page")

# Assigning key
walkscore_raw = tables[0]

# Confirm correct table selected
print('Confirmed raw shape:', walkscore_raw.shape)
print('\nFirst 5 rows of raw scraped data:')
display(walkscore_raw.head())

Found 1 table(s) on the page
Confirmed raw shape: (130, 6)

First 5 rows of raw scraped data:


,City,State,Walk Score,Transit Score,Bike Score,Population
0,Gatineau,CA-QC,37.4,40.8,57.6,265349
1,Longueuil,CA-QC,54.4,52.5,69.6,231409
2,QuÃ©bec,CA-QC,44.6,46.9,59.3,516622
3,MontrÃ©al,CA-QC,65.4,67.0,72.6,1649519
4,Laval,CA-QC,43.1,46.3,57.3,401553


# Step #1 – Clean and rename column headers for consistency

**Making all table headers lower-case to ensure consistency**

In [12]:
# Copy for reproduceability
walkscore = walkscore_raw.copy()

# Data information before step to clean for comparison
original_cols = list(walkscore.columns)

walkscore.columns = [
    'city', 'state', 'walk_score', 'transit_score',
    'bike_score', 'population'
]

# Change Summary
changed_headers = sum(
    1 for a,
    b in zip(
        original_cols,
        walkscore.columns) if a != b)

print('Step #1 – Standardized column names:')
display(walkscore.head(3))

print(
    f"Change count: {changed_headers} out of {len(original_cols)} headers were changed.")
print(f"Changes: Column names were made lowercase for consistency")

Step #1 – Standardized column names:


,city,state,walk_score,transit_score,bike_score,population
0,Gatineau,CA-QC,37.4,40.8,57.6,265349
1,Longueuil,CA-QC,54.4,52.5,69.6,231409
2,QuÃ©bec,CA-QC,44.6,46.9,59.3,516622


Change count: 6 out of 6 headers were changed.
Changes: Column names were made lowercase for consistency


# Step #2 – Fix mangled letter and characters in city names

**Correcting the random characters that returned in some of the city names by updating encoding from latin1 to UTF-8**

In [13]:
# Before fix – count how many look garbled (terminology may need
# adjusted, lookup terms before submitting)
garbled_before = walkscore['city'].str.contains(
    r'[ÃâÃ©Ã¨]', regex=True, na=False).sum()

# Change emcoding from latin1 to UTF-8
walkscore['city'] = walkscore['city'].str.encode(
    'latin1').str.decode('utf-8', errors='replace')

# After fix
garbled_after = walkscore['city'].str.contains(
    r'[ÃâÃ©Ã¨]', regex=True, na=False).sum()

# Change summary
print('Step #2 – Corrected encoding in city names:')
display(walkscore[walkscore['city'].str.contains(
    'Montr|Québec|é', na=False)].head(6))

print(
    f"Change count: {garbled_before} cities had visibly garbled characters before; "
    f"{garbled_before - garbled_after} were fixed.")
print(f"Changes: City names with garbled characters were re-encoded "
      f"from latin1 to utf-8 to restore proper accented letters (changed QuÃ©bec to Québec).")

Step #2 – Corrected encoding in city names:


,city,state,walk_score,transit_score,bike_score,population
2,Québec,CA-QC,44.6,46.9,59.3,516622
3,Montréal,CA-QC,65.4,67.0,72.6,1649519


Change count: 2 cities had visibly garbled characters before; 2 were fixed.
Changes: City names with garbled characters were re-encoded from latin1 to utf-8 to restore proper accented letters (changed QuÃ©bec to Québec).


# Step #3 – Convert scores and population to numeric

**Making sure values are changed to numeric for consistency, updated missing values to NaN**

In [14]:
# Convert to numeric
numeric_cols = ['walk_score', 'transit_score', 'bike_score', 'population']

# Count non-numeric values before conversion
non_numeric_before = {}
for col in numeric_cols:
    non_numeric_before[col] = walkscore[col].apply(
        lambda x: not str(x).replace('.', '', 1).isdigit()).sum()

for col in numeric_cols:
    walkscore[col] = pd.to_numeric(walkscore[col], errors='coerce')

# Count how many became NaN due to conversion
nan_now = walkscore[numeric_cols].isna().sum()

print('Step #3 – Ensured numeric types. Missing values count:')
print(nan_now)
print('\nBasic stats after conversion:')
display(walkscore[numeric_cols].describe())

print('Change count:')
for col in numeric_cols:
    print(
        f"  - {col}: {non_numeric_before[col]} non-numeric values detected → "
        f"{nan_now[col]} changed to NaN")
print(
    'Changes: Score and population columns were converted from object/string '
    'to numeric types; invalid entries were changed to NaN.')

Step #3 – Ensured numeric types. Missing values count:
walk_score       0
transit_score    7
bike_score       0
population       0
dtype: int64

Basic stats after conversion:


,walk_score,transit_score,bike_score,population
count,130.0,123.0,130.0,130.0
mean,47.8,38.8,52.3,579269.3
std,15.3,16.0,11.8,843844.3
min,21.3,0.3,29.2,200564.0
25%,37.3,26.6,43.1,236099.0
50%,44.4,35.7,52.2,345656.0
75%,56.8,48.7,59.8,600956.0
max,88.7,88.6,83.5,8175133.0


Change count:
  - walk_score: 0 non-numeric values detected → 0 changed to NaN
  - transit_score: 7 non-numeric values detected → 7 changed to NaN
  - bike_score: 0 non-numeric values detected → 0 changed to NaN
  - population: 0 non-numeric values detected → 0 changed to NaN
Changes: Score and population columns were converted from object/string to numeric types; invalid entries were changed to NaN.


# Step #4 – Add country column and clean state codes

**Moving the Canadian indicator from being a prefic on the `state` code, to a new column named `country`**

In [15]:
# Count the CA prefix for summary prior to updating
ca_prefix_before = walkscore['state'].str.startswith('CA-').sum()

# Creating label and labeling CA as Canada all others as United States
walkscore['country'] = walkscore['state'].apply(
    lambda x: 'Canada' if str(x).startswith('CA-') else 'United States'
)
walkscore['state_code'] = walkscore['state'].str.replace(
    'CA-', '', regex=False)

print('Step #4 – Added country and cleaned state codes:')
display(walkscore[['city', 'state', 'country', 'state_code']].head(10))

print(
    f"Change count: {ca_prefix_before} rows had 'CA-' prefix removed from state.")
print(f"Changes: New 'country' column added (US vs Canada); 'CA-' prefix "
      f"removed from Canadian entries to create clean 'state_code'.")

Step #4 – Added country and cleaned state codes:


,city,state,country,state_code
0,Gatineau,CA-QC,Canada,QC
1,Longueuil,CA-QC,Canada,QC
2,Québec,CA-QC,Canada,QC
3,Montréal,CA-QC,Canada,QC
4,Laval,CA-QC,Canada,QC
5,Windsor,CA-ON,Canada,ON
6,Kitchener,CA-ON,Canada,ON
7,Ottawa,CA-ON,Canada,ON
8,Vaughan,CA-ON,Canada,ON
9,Hamilton,CA-ON,Canada,ON


Change count: 22 rows had 'CA-' prefix removed from state.
Changes: New 'country' column added (US vs Canada); 'CA-' prefix removed from Canadian entries to create clean 'state_code'.


# Step #5 – Flag rows with missing score

**Adding column that will flag any rows with a missing value**

In [16]:
# Find missing values
walkscore['score_missing_flag'] = walkscore[
    ['walk_score', 'transit_score', 'bike_score']
].isna().any(axis=1)

# Count missing values for summary
missing_count = walkscore['score_missing_flag'].sum()

print('Step #5 – Flagged rows missing any score:')
print(walkscore['score_missing_flag'].value_counts())
print('\nExample rows with missing scores:')
display(walkscore[walkscore['score_missing_flag']].head(5))

print(
    f"Change count: {missing_count} rows flagged as having at least one missing score.")
print(f"Changes: New column added to mark rows with any missing values.")

Step #5 – Flagged rows missing any score:
score_missing_flag
False    123
True       7
Name: count, dtype: int64

Example rows with missing scores:


,city,state,walk_score,transit_score,bike_score,population,country,state_code,score_missing_flag
55,Baton Rouge,LA,39.1,NaN,44.3,229493,United States,LA,True
69,Richmond,VA,50.9,NaN,50.8,204214,United States,VA,True
75,Toledo,OH,46.4,NaN,45.6,287208,United States,OH,True
85,Laredo,TX,36.8,NaN,39.6,236091,United States,TX,True
100,Greensboro,NC,29.4,NaN,32.2,269666,United States,NC,True


Change count: 7 rows flagged as having at least one missing score.
Changes: New column added to mark rows with any missing values.


# Step #6 – Remove any duplicates, sort descending by Walk Score, and reset index

**Removing duplicate values, though I do not believe there will be any, sorting the results from highest score to lowest, and resetting the index**

In [17]:
# Count duplicates prior to removing for summary
dupe_count = walkscore.duplicated(subset=['city', 'state_code']).sum()

# Remove Duplicate Data
walkscore = walkscore.drop_duplicates(subset=['city', 'state_code'])
walkscore = walkscore.sort_values(
    'walk_score',
    ascending=False).reset_index(
        drop=True)

print('Step #6 – Removed duplicates and sorted by Walk Score descending')
print('Final cleaned shape:', walkscore.shape)
print('\nTop 10 most walkable large cities:')
display(walkscore.head(10))

print(f"Change count: {dupe_count} duplicate rows removed.")
print(
    f"Changes: Verified no duplicates exist, sorted results highest to lowest by walk score "
    f"and reset index for clean row numbering.")

Step #6 – Removed duplicates and sorted by Walk Score descending
Final cleaned shape: (130, 9)

Top 10 most walkable large cities:


,city,state,walk_score,transit_score,bike_score,population,country,state_code,score_missing_flag
0,San Francisco,CA,88.7,77.1,72.3,805235,United States,CA,False
1,New York,NY,88.0,88.6,69.3,8175133,United States,NY,False
2,Jersey City,NJ,86.6,70.5,63.9,247597,United States,NJ,False
3,Boston,MA,82.8,72.4,69.4,617594,United States,MA,False
4,Vancouver,CA-BC,79.8,74.4,78.9,603502,Canada,BC,False
5,Chicago,IL,77.2,65.0,72.2,2695598,United States,IL,False
6,Washington D.C.,DC,76.7,68.7,69.5,601723,United States,DC,False
7,Miami,FL,76.6,57.0,64.0,399457,United States,FL,False
8,Newark,NJ,75.9,65.0,51.1,277140,United States,NJ,False
9,Oakland,CA,75.3,56.6,65.5,390724,United States,CA,False


Change count: 0 duplicate rows removed.
Changes: Verified no duplicates exist, sorted results highest to lowest by walk score and reset index for clean row numbering.


In [18]:
# Dataset after transformations completed

print('\nFinal cleaned dataset (first 15 rows):')
display(walkscore.head(15))


Final cleaned dataset (first 15 rows):


,city,state,walk_score,transit_score,bike_score,population,country,state_code,score_missing_flag
0,San Francisco,CA,88.7,77.1,72.3,805235,United States,CA,False
1,New York,NY,88.0,88.6,69.3,8175133,United States,NY,False
2,Jersey City,NJ,86.6,70.5,63.9,247597,United States,NJ,False
3,Boston,MA,82.8,72.4,69.4,617594,United States,MA,False
4,Vancouver,CA-BC,79.8,74.4,78.9,603502,Canada,BC,False
5,Chicago,IL,77.2,65.0,72.2,2695598,United States,IL,False
6,Washington D.C.,DC,76.7,68.7,69.5,601723,United States,DC,False
7,Miami,FL,76.6,57.0,64.0,399457,United States,FL,False
8,Newark,NJ,75.9,65.0,51.1,277140,United States,NJ,False
9,Oakland,CA,75.3,56.6,65.5,390724,United States,CA,False


## Transformation Summary

The table below summarizes the number of changes / items affected during each transformation step.

| Step | Description                                      | Changes Made / Items Affected                          |
|------|--------------------------------------------------|--------------------------------------------------------|
| 1    | Standardized column headers to snake_case        | 6 out of 6 headers renamed                             |
| 2    | Fixed UTF-8 encoding issues in city names        | 2 garbled city names fixed                             |
| 3    | Converted scores & population to numeric types   | 7 values coerced to NaN (all in transit_score contaning "--")         |
| 4    | Added country column & cleaned state codes       | 22 'CA-' prefixes removed from state                    |
| 5    | Flagged rows with missing scores                 | 7 rows flagged as having at least one missing score    |
| 6    | Removed duplicates & sorted by Walk Score        | 0 duplicate rows removed                               |


## Ethical Implications of Data Wrangling – Walk Score Website Data

The transformations applied to the Walk Score city rankings dataset included standardizing column headers, correcting UTF‑8 encoding issues in city names, converting score and population fields to numeric types (coercing invalid entries such as “--” to `NaN`), adding a country column and cleaning state codes, flagging rows with missing scores, and removing duplicates while sorting by Walk Score. These steps improved consistency, readability, and analytical usability while preserving the original meaning of the data. No transformations altered the underlying walkability scores or their relative rankings.

Walk Score publishes this aggregated city‑level data openly for public and research use, and no legal or regulatory restrictions limit its analysis. However, ethical considerations remain. Walkability metrics can be misinterpreted as neutral indicators of city quality, when in reality they reflect long‑standing patterns of urban planning, infrastructure investment, zoning decisions, and historical policies that often correlate with socioeconomic and racial disparities. Presenting the cleaned dataset without acknowledging this context could unintentionally reinforce inequitable narratives about “good” or “bad” cities.

Key assumptions included treating comma‑containing city names as valid entries and interpreting out‑of‑range or non‑numeric scores as errors. These assumptions are reasonable but not infallible. To mitigate risks, all transformations were fully documented, original values were preserved where possible, and missing data was flagged rather than imputed or removed. Future analysis should incorporate socioeconomic, historical, and policy context to avoid oversimplifying structural factors that shape walkability outcomes.

<div style="border: 5px solid black; padding: 10px;">

# **Name:** Tim Hollis  
# **Course:** DSC540 - Data Preparation  
# **Date:**  02/01/2026
# **Assignment:** Milestone 2
## HUD Fair Market Rent (FMR) Dataset – 2024 Flat File
## Data Source: HUD User – Fair Market Rents


### Initial Setup

In [19]:
# Import Libraries
import pandas as pd
import numpy as np

# Load the HUD FMR flat file (2024 CSV)
fmr_raw = pd.read_csv('FMR Data - Flat File for Final.csv')

# Preview the raw dataset
display(fmr_raw.head())

# Create a working copy for transformations
fmr = fmr_raw.copy()

,stusps,state,hud_area_code,countyname,county_town_name,metro,hud_area_name,fips,pop2020,fmr_0,fmr_1,fmr_2,fmr_3,fmr_4
0,AL,1,METRO33860M33860,Autauga County,NaN,1,"Montgomery, AL MSA",100199999,55639,836,913,1092,1383,1753
1,AL,1,METRO19300M19300,Baldwin County,NaN,1,"Daphne-Fairhope-Foley, AL MSA",100399999,218289,1051,1056,1362,1670,2114
2,AL,1,NCNTY01005N01005,Barbour County,NaN,0,"Barbour County, AL",100599999,25026,652,656,857,1089,1141
3,AL,1,METRO13820M13820,Bibb County,NaN,1,"Birmingham-Hoover, AL HUD Metro FMR Area",100799999,22374,983,1109,1245,1570,1752
4,AL,1,METRO13820M13820,Blount County,NaN,1,"Birmingham-Hoover, AL HUD Metro FMR Area",100999999,57755,983,1109,1245,1570,1752


## Step 1 – Standardize Column Headers

To ensure consistency and make downstream transformations easier to manage, the column headers were standardized by trimming whitespace, converting all names to lowercase, and replacing spaces and hyphens with underscores. The number of headers that changed during this process was also recorded for transparency.

In [20]:
# Step 1: Standardize column headers and count changes
old_cols = fmr.columns.tolist()

fmr.columns = (
    fmr.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

new_cols = fmr.columns.tolist()

# Count how many column names changed
header_changes = sum([1 for old, new in zip(old_cols, new_cols) if old != new])
header_changes

0

## Step 2: Normalize County and State Names

The HUD dataset uses the column `countyname` for county names and `state` for state abbreviations. This step standardizes county names to title case and state abbreviations to uppercase. A count of how many values changed is included for transparency.

In [21]:
# Step 2: Normalize county and state names and count edits

# Convert to string first to avoid .str accessor errors
fmr['countyname'] = fmr['countyname'].astype(str)
fmr['state'] = fmr['state'].astype(str)

county_before = fmr['countyname'].copy()
state_before = fmr['state'].copy()

# Apply casing fixes
fmr['countyname'] = fmr['countyname'].str.title()
fmr['state'] = fmr['state'].str.upper()

# Count changes
county_changes = (county_before != fmr['countyname']).sum()
state_changes = (state_before != fmr['state']).sum()

county_changes, state_changes

(np.int64(103), np.int64(0))

## Step 3: Remove Duplicate Rows

Duplicate entries may appear due to repeated county listings or formatting inconsistencies in the HUD dataset. This step identifies and removes duplicate rows to ensure each county appears only once. The number of duplicates removed is reported for transparency and data quality validation.

In [22]:
# Step 3: Remove duplicate rows and count how many were removed
duplicate_count = fmr.duplicated().sum()

# Drop duplicates
fmr = fmr.drop_duplicates()

duplicate_count

np.int64(0)

## Step 4: Convert Rent Columns to Numeric

Some Fair Market Rent (FMR) values may be stored as strings or contain formatting characters such as commas or dollar signs. This step converts all FMR-related columns to numeric values so they can be used in calculations and statistical analysis. A count of how many values changed during conversion is included for transparency.

In [23]:
# Step 4: Convert rent columns to numeric and count changes

# Identify all FMR columns (fmr_0, fmr_1, fmr_2, fmr_3, fmr_4)
rent_cols = [col for col in fmr.columns if col.startswith('fmr_')]

conversion_count = 0

for col in rent_cols:
    before = fmr[col].copy()

    # Convert to string, remove formatting, convert to float
    fmr[col] = (
        fmr[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('$', '', regex=False)
        .astype(float)
    )

    # Count how many values changed
    conversion_count += (before != fmr[col]).sum()

conversion_count

np.int64(0)

## Step 5: Identify Outliers in 2-Bedroom FMR Values

Outliers can indicate unusual market conditions, data entry issues, or extreme housing cost variations. This step uses the Interquartile Range (IQR) method to detect outliers in the 2-bedroom Fair Market Rent (FMR) column. The number of outliers identified is reported for transparency and data quality assessment.

In [24]:
# Step 5: Identify outliers in 2-bedroom FMR and count them

# Calculate IQR boundaries
q1 = fmr['fmr_2'].quantile(0.25)
q3 = fmr['fmr_2'].quantile(0.75)
iqr = q3 - q1

lower_outlier = q1 - 1.5 * iqr
upper_outlier = q3 + 1.5 * iqr

# Identify outliers
outliers = fmr[(fmr['fmr_2'] < lower_outlier) | (fmr['fmr_2'] > upper_outlier)]

# Count outliers
outlier_count = len(outliers)
outlier_count

213

**Note:** A total of 213 outliers were detected in the 2-bedroom FMR values. These outliers were identified but not removed, as the assignment requires detection only. High-cost housing markets commonly produce extreme values in HUD datasets.

## Final Cleaned Dataset

The following preview displays the cleaned and formatted dataset after all transformation steps were applied. This allows verification that the dataset is structured correctly and ready for analysis in later milestones.

| 5    | Identified outliers in 2-bedroom FMR      | `outlier_count` |

In [25]:
# Display the first 20 rows of the cleaned dataset
fmr.head(20)

,stusps,state,hud_area_code,countyname,county_town_name,metro,hud_area_name,fips,pop2020,fmr_0,fmr_1,fmr_2,fmr_3,fmr_4
0,AL,1,METRO33860M33860,Autauga County,NaN,1,"Montgomery, AL MSA",100199999,55639,836.0,913.0,1092.0,1383.0,1753.0
1,AL,1,METRO19300M19300,Baldwin County,NaN,1,"Daphne-Fairhope-Foley, AL MSA",100399999,218289,1051.0,1056.0,1362.0,1670.0,2114.0
2,AL,1,NCNTY01005N01005,Barbour County,NaN,0,"Barbour County, AL",100599999,25026,652.0,656.0,857.0,1089.0,1141.0
3,AL,1,METRO13820M13820,Bibb County,NaN,1,"Birmingham-Hoover, AL HUD Metro FMR Area",100799999,22374,983.0,1109.0,1245.0,1570.0,1752.0
4,AL,1,METRO13820M13820,Blount County,NaN,1,"Birmingham-Hoover, AL HUD Metro FMR Area",100999999,57755,983.0,1109.0,1245.0,1570.0,1752.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,AL,1,NCNTY01031N01031,Coffee County,NaN,0,"Coffee County, AL",103199999,52238,713.0,725.0,952.0,1342.0,1472.0
16,AL,1,METRO22520M22520,Colbert County,NaN,1,"Florence-Muscle Shoals, AL MSA",103399999,54957,664.0,720.0,940.0,1157.0,1425.0
17,AL,1,NCNTY01035N01035,Conecuh County,NaN,0,"Conecuh County, AL",103599999,12219,642.0,666.0,857.0,1208.0,1283.0
18,AL,1,NCNTY01037N01037,Coosa County,NaN,0,"Coosa County, AL",103799999,10696,658.0,683.0,879.0,1063.0,1315.0


## Transformation Summary

The table below summarizes the number of changes made during each transformation step.

| Step | Description                               | Changes Made |
|------|-------------------------------------------|--------------|
| 1    | Standardized column headers               | `header_changes` |
| 2    | Normalized county and state names         | `county_changes` county changes, `state_changes` state changes |
| 3    | Removed duplicate rows                    | `duplicate_count` |
| 4    | Converted rent columns to numeric         | `conversion_count` |

## Ethical Implications of Data Wrangling

The transformations applied to the HUD Fair Market Rent (FMR) dataset were designed to improve clarity, consistency, and analytical usability. These steps included standardizing column names, normalizing geographic fields, removing duplicates, converting rent values to numeric formats, and identifying outliers.

While these actions enhance data quality, they also introduce ethical considerations. For example, identifying outliers may highlight extreme housing costs, but removing or altering them could distort the lived realities of communities facing unusually high or low rents. Similarly, standardizing geographic names improves consistency but must be done carefully to avoid overwriting meaningful local distinctions. 

Because HUD data reflects real socioeconomic conditions, transparency in each transformation is essential to avoid misrepresenting affordability or housing disparities. All transformations were documented, and no values were removed or altered beyond formatting (which technically was not required on this dataset, as the data was already in the formats I needed), ensuring the dataset remains faithful to its original meaning while becoming more accessible for analysis.

<div style="border: 5px solid black; padding: 10px;">

# **Name:** Tim Hollis  
# **Course:** DSC540 - Data Preparation  
# **Date:**  01/18/2026
# **Assignment:** Milestone 1 - Data Selection/Plan Submission


# Milestone 1 – Project Plan

## Project Subject Area

**How transportation access and basic living costs vary across U.S. cities and how these factors relate to overall quality of life.**

## Data Sources

### Flat File
- **Data Source**: HUD Fair Market Rent (FMR) Data  
- **Description**: Annual rent estimates for 0–4 bedroom units across U.S. counties and metro areas. Includes geographic identifiers such as county names and FIPS codes. This dataset provides a cost-of-living proxy and contains natural inconsistencies in naming and formatting.  
- **Link**: [HUD Fair Market Rent Datasets](https://www.huduser.gov/portal/datasets/fmr.html)

### API
- **Data Source**: U.S. Census Bureau API (ACS Data)  
- **Description**: Provides demographic and socioeconomic indicators such as median household income, commute time, poverty rates, and population characteristics. Data is returned in JSON format and requires reshaping and cleaning.  
- **Link**: [Census ACS 2021 Profile Groups (DP05)](https://api.census.gov/data/2021/acs/acs1/profile/groups/DP05.html)

### Website Table
- **Data Source**: Walk Score City Rankings  
- **Description**: City-level walkability, transit access, and bikeability scores presented in an HTML table. City names often differ from other sources, and some cities have missing or partial scores.  
- **Link**: [Walk Score Cities and Neighborhoods](https://www.walkscore.com/cities-and-neighborhoods/)

## Relationships Between the Data Sources

All three datasets can be connected at the city level, but each uses different naming conventions and geographic identifiers.  

- HUD data is organized by county and metro area  
- Walk Score uses city names  
- Census data uses place-level identifiers and FIPS codes  

To create a relationship between them, I will:  
- Standardize city names  
- Map counties to cities when necessary  
- Use FIPS codes where available  
- Create a custom mapping table to align city names across sources when direct matches do not exist  

This process will require cleaning, normalizing text fields, and resolving mismatches.

## Project Approach and Plan

The goal of this project is to integrate three distinct datasets:  
- HUD Fair Market Rent data  
- Walk Score transportation metrics  
- Census API demographic indicators  

to explore how transportation access and basic living costs relate to quality of life across U.S. cities.

My approach will focus heavily on data wrangling:
- Cleaning inconsistent geographic identifiers  
- Reshaping tables  
- Handling missing values  
- Merging datasets that were not designed to work together  

**Planned steps**:  
1. Load each dataset in its raw form and document structure and geographic levels  
2. Standardize city names and create a unified geographic key (likely using FIPS codes where possible)  
3. Merge the datasets after establishing consistent identifiers  
4. Engineer new variables that capture affordability, accessibility, and demographic context  

## Concerns and Challenges

The primary challenge will be reconciling geographic inconsistencies:  
- HUD data is county-based  
- Walk Score is city-based  
- Census data can be retrieved at multiple geographic levels  

City names may not match exactly across sources, and some cities may appear in one dataset but not another.  

Additional challenges include:  
- Nested JSON structure from the Census API requiring reshaping  
- Missing values  
- Inconsistent formatting  
- Ambiguous geographic boundaries  

These challenges are expected and align with the project’s emphasis on data manipulation.

## Ethical Implications

Quality-of-life data can unintentionally reinforce biases if interpreted without context. Cities with lower walkability or higher housing costs may also face structural inequalities, historical underinvestment, or demographic disparities. It is important to avoid framing these differences as deficiencies of the communities themselves.

Additionally, using demographic data requires sensitivity to privacy and representation, even when working with aggregated public datasets.  

My focus will remain on understanding structural patterns rather than making value judgments about specific cities or populations.